# School Aggregations
- [School Aggregations](#school-aggregations)
- [Nearest Schools](#nearest-schools)
- [Primary Schools](#primary-schools)
- [Secondary Schools](#secondary-schools)
- [Tertiary Schools](#tertiary-schools)
- [Conclusion](#conclusion)

Written By: Keltonn [A0222591J]

AI Used: Claude Sonnet 5.5 for API Lookup, Data Scraping

In this notebook, we will be doing feature engineering for the School features.

### Feature Types

There are two general types of features for this problem:

|Feature Type|Description|
|---|---|
| Temporal Features | Features that change every month, like stock market data and Economic Indicators |
| Static Features | Features that have no trend, don't change. Includes most location features, such as housing type |

### Task

For this notebook, we will be processing the features related to the nearby schools for each HDB Block. This is part of the static feature type as the nearby schools will not change over time for the period of the training and test dataset.

There are 4 kinds of schools that Claude has scraped from the MOE website metadata:

|School Type|Description|
|---|---|
| Primary School | First school most kids will go through, contains no entry score, usually most important factor in housing |
| Secondary School |  |
| Tertiary | We only consider Junior Colleges here, no Polytechnics |
| Mixed | Special schools which combine both Secondary and Tertiary education |

In [8]:
import pandas as pd
import numpy as np
import scipy as scp
import math
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import joblib
ROOT = '/Users/keltonn/Desktop/Projects/housing_rent_price/final_report'

# Nearest Schools
- [School Aggregations](#school-aggregations)
- [Nearest Schools](#nearest-schools)
- [Primary Schools](#primary-schools)
- [Secondary Schools](#secondary-schools)
- [Tertiary Schools](#tertiary-schools)
- [Conclusion](#conclusion)

We take the nearest schools

In [4]:
LAT0 = 1.35
R_EARTH = 6371000.0
def to_xy(lat, lon):
    x = np.radians(np.asarray(lon, float)) * math.cos(math.radians(LAT0)) * R_EARTH
    y = np.radians(np.asarray(lat, float)) * R_EARTH
    return np.column_stack([np.ravel(x), np.ravel(y)])

In [3]:
schools = pd.read_csv(f'{ROOT}/data/original_dataset/auxiliary/sg-schools.csv')[['NAME','TYPE_CODE','NATRUE_CODE','MAINLEVEL_CODE','LATITUDE','LONGITUDE']]
houses = pd.read_csv(f'{ROOT}/data/original_dataset/auxiliary/sg-hdb-block.csv')[['TOWN','BLOCK','STREET','POSTAL_CODE','LATITUDE','LONGITUDE']]
schools.columns = ['nearest_school','nearest_school_type','nearest_school_nature','nearest_school_level','LATITUDE','LONGITUDE']
sch_type = schools[['nearest_school','nearest_school_type','nearest_school_nature']]
display(schools)
display(houses)

,nearest_school,nearest_school_type,nearest_school_nature,nearest_school_level,LATITUDE,LONGITUDE
0,admiralty primary school,government school,co-ed school,primary,1.442635,103.800040
1,admiralty secondary school,government school,co-ed school,secondary (s1-s5),1.445891,103.802398
2,ahmad ibrahim primary school,government school,co-ed school,primary,1.433153,103.832942
3,ahmad ibrahim secondary school,government school,co-ed school,secondary (s1-s5),1.436060,103.829719
4,ai tong school,government-aided sch,co-ed school,primary,1.360583,103.833020
...,...,...,...,...,...,...
332,zhangde primary school,government school,co-ed school,primary,1.284212,103.825952
333,zhenghua primary school,government school,co-ed school,primary,1.379549,103.769314
334,zhenghua secondary school,government school,co-ed school,secondary (s1-s5),1.388366,103.765511
335,zhonghua primary school,government school,co-ed school,primary,1.360203,103.869762


,TOWN,BLOCK,STREET,POSTAL_CODE,LATITUDE,LONGITUDE
0,serangoon,106,serangoon north avenue 1,550106,1.370245,103.870823
1,bukit merah,34,telok blangah way,090034,1.273518,103.822481
2,punggol,226b,sumang lane,822226,1.402685,103.893609
3,marine parade,74,marine drive,440074,1.303705,103.907214
4,choa chu kang,284,choa chu kang avenue 3,NIL,1.379714,103.742138
...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,310155,1.332796,103.845589
9460,sembawang,307,canberra road,750307,1.445923,103.821657
9461,woodlands,681a,woodlands drive 62,731681,1.439534,103.803797
9462,choa chu kang,673a,choa chu kang crescent,681673,1.401014,103.747524


In [5]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(schools.LATITUDE.values, schools.LONGITUDE.values)

# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

count_rows = []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_school": schools.nearest_school[nearest],
        "nearest_school_m": round(float(d[nearest]), 1),
    }

    count_rows.append(results)

counts = pd.DataFrame(count_rows)
counts = pd.merge(counts, schools[['nearest_school', 'nearest_school_type', 'nearest_school_nature',
       'nearest_school_level',]], how = 'left', on = 'nearest_school')
counts

,TOWN,BLOCK,STREET,nearest_school,nearest_school_m,nearest_school_type,nearest_school_nature,nearest_school_level
0,serangoon,106,serangoon north avenue 1,serangoon garden secondary school,200.6,government school,co-ed school,secondary (s1-s5)
1,bukit merah,34,telok blangah way,chij st. theresa's convent,204.0,government-aided sch,girls' school,secondary (s1-s5)
2,punggol,226b,sumang lane,yusof ishak secondary school,558.0,government school,co-ed school,secondary (s1-s5)
3,marine parade,74,marine drive,chij (katong) primary,531.7,government-aided sch,girls' school,primary
4,choa chu kang,284,choa chu kang avenue 3,chua chu kang primary school,220.0,government school,co-ed school,primary
...,...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,chij secondary (toa payoh),260.3,government-aided sch,girls' school,secondary (s1-s5)
9460,sembawang,307,canberra road,sembawang primary school,120.3,government school,co-ed school,primary
9461,woodlands,681a,woodlands drive 62,greenwood primary school,110.9,government school,co-ed school,primary
9462,choa chu kang,673a,choa chu kang crescent,unity secondary school,200.6,government school,co-ed school,secondary (s1-s5)


In [7]:
mx =  pd.read_csv(f'{ROOT}/data/features/raw/collected_data/school_data/sg-schools-mixed.csv')
mx_sec = mx[['NAME', 'LATITUDE', 'LONGITUDE','pg3_min', 'pg3_max',]]
mx_jc = mx[['NAME', 'LATITUDE', 'LONGITUDE', 
       'l1r5_international_baccalaureate_min',
       'l1r5_international_baccalaureate_max',
       'l1r5_arts_min', 'l1r5_arts_max', 
       'l1r5_science_min', 'l1r5_science_max',]]
mx_jc['l1r5_arts_max'] = mx_jc['l1r5_arts_max'].fillna(mx_jc['l1r5_international_baccalaureate_max'])
mx_jc['l1r5_arts_min'] = mx_jc['l1r5_arts_min'].fillna(mx_jc['l1r5_international_baccalaureate_min'])
mx_jc['l1r5_science_max'] = mx_jc['l1r5_science_max'].fillna(mx_jc['l1r5_international_baccalaureate_max'])
mx_jc['l1r5_science_min'] = mx_jc['l1r5_science_min'].fillna(mx_jc['l1r5_international_baccalaureate_min'])
mx_jc = mx_jc.drop(['l1r5_international_baccalaureate_min','l1r5_international_baccalaureate_max'], axis = 1)
display(mx_sec)
display(mx_jc)

/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipykernel_6789/2686926115.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mx_jc['l1r5_arts_max'] = mx_jc['l1r5_arts_max'].fillna(mx_jc['l1r5_international_baccalaureate_max'])
/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipykernel_6789/2686926115.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mx_jc['l1r5_arts_min'] = mx_jc['l1r5_arts_min'].fillna(mx_jc['l1r5_international_baccalaureate_min'])
/var/folders/j9/6y3df2850854pj09qr79r10w0000gn/T/ipyke

,NAME,LATITUDE,LONGITUDE,pg3_min,pg3_max
0,anglo-chinese school (independent),1.303704,103.780363,7,13
1,catholic high school,1.354525,103.844901,6(D),8(M)
2,chij st. nicholas girls' school,1.373477,103.834253,4(D),7(M)
3,dunman high school,1.299962,103.882942,4(D),8(M)
4,hwa chong institution,1.324948,103.805704,4(D),6(M)
5,maris stella high school,1.341395,103.877852,7(M),12
6,national junior college,1.330293,103.804111,5,8
7,nus high school of mathematics and science,1.306007,103.770153,NaN,NaN
8,raffles institution,1.346188,103.845875,4,6
9,river valley high school,1.344078,103.709022,4(M),9(M)


,NAME,LATITUDE,LONGITUDE,l1r5_arts_min,l1r5_arts_max,l1r5_science_min,l1r5_science_max
0,anglo-chinese school (independent),1.303704,103.780363,2.0,5.0,2.0,5.0
1,catholic high school,1.354525,103.844901,NaN,NaN,NaN,NaN
2,chij st. nicholas girls' school,1.373477,103.834253,NaN,NaN,NaN,NaN
3,dunman high school,1.299962,103.882942,6.0,8.0,4.0,6.0
4,hwa chong institution,1.324948,103.805704,2.0,5.0,2.0,3.0
5,maris stella high school,1.341395,103.877852,NaN,NaN,NaN,NaN
6,national junior college,1.330293,103.804111,4.0,8.0,2.0,6.0
7,nus high school of mathematics and science,1.306007,103.770153,NaN,NaN,NaN,NaN
8,raffles institution,1.346188,103.845875,2.0,5.0,2.0,3.0
9,river valley high school,1.344078,103.709022,7.0,9.0,2.0,8.0


# Primary Schools
- [School Aggregations](#school-aggregations)
- [Nearest Schools](#nearest-schools)
- [Primary Schools](#primary-schools)
- [Secondary Schools](#secondary-schools)
- [Tertiary Schools](#tertiary-schools)
- [Conclusion](#conclusion)

In [10]:
pri = pd.read_csv(f'{ROOT}/data/features/raw/collected_data/school_data/sg-schools-primary.csv')
pri = pri[['NAME', 'LATITUDE', 'LONGITUDE',]]
pri

,NAME,LATITUDE,LONGITUDE
0,admiralty primary school,1.442635,103.800040
1,ahmad ibrahim primary school,1.433153,103.832942
2,ai tong school,1.360583,103.833020
3,alexandra primary school,1.291334,103.824425
4,anchor green primary school,1.390370,103.887165
...,...,...,...
174,yuhua primary school,1.342802,103.741106
175,yumin primary school,1.351292,103.950462
176,zhangde primary school,1.284212,103.825952
177,zhenghua primary school,1.379549,103.769314


In [11]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(pri.LATITUDE.values, pri.LONGITUDE.values)

# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

count_rows = []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_pri_school": pri.NAME[nearest],
        "nearest_pri_school_m": round(float(d[nearest]), 1),
    }

    count_rows.append(results)

pri_counts = pd.DataFrame(count_rows)
sch_type2 = sch_type.copy(deep=True)
sch_type2.columns = ['nearest_pri_school', 'pri_sch_type', 'pri_sch_nature']
pri_counts = pd.merge(pri_counts, sch_type2, how = 'left', on = ['nearest_pri_school'])
pri_counts

,TOWN,BLOCK,STREET,nearest_pri_school,nearest_pri_school_m,pri_sch_type,pri_sch_nature
0,serangoon,106,serangoon north avenue 1,rosyth school,549.8,government school,co-ed school
1,bukit merah,34,telok blangah way,radin mas primary school,248.3,government school,co-ed school
2,punggol,226b,sumang lane,punggol green primary school,584.2,government school,co-ed school
3,marine parade,74,marine drive,chij (katong) primary,531.7,government-aided sch,girls' school
4,choa chu kang,284,choa chu kang avenue 3,chua chu kang primary school,220.0,government school,co-ed school
...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,chij primary (toa payoh),416.0,government-aided sch,girls' school
9460,sembawang,307,canberra road,sembawang primary school,120.3,government school,co-ed school
9461,woodlands,681a,woodlands drive 62,greenwood primary school,110.9,government school,co-ed school
9462,choa chu kang,673a,choa chu kang crescent,unity primary school,219.0,government school,co-ed school


# Secondary Schools
- [School Aggregations](#school-aggregations)
- [Nearest Schools](#nearest-schools)
- [Primary Schools](#primary-schools)
- [Secondary Schools](#secondary-schools)
- [Tertiary Schools](#tertiary-schools)
- [Conclusion](#conclusion)

In [12]:
sec =  pd.read_csv(f'{ROOT}/data/features/raw/collected_data/school_data/sg-schools-secondary.csv')
sec = sec[['NAME', 'LATITUDE', 'LONGITUDE','pg3_min', 'pg3_max',]]
sec = pd.concat([sec, mx_sec], axis = 0).reset_index(drop = True)
sec

,NAME,LATITUDE,LONGITUDE,pg3_min,pg3_max
0,admiralty secondary school,1.445891,103.802398,16,22
1,ahmad ibrahim secondary school,1.436060,103.829719,11,18
2,anderson secondary school,1.374340,103.851554,7,11
3,ang mo kio secondary school,1.367337,103.842155,12,16
4,anglican high school,1.331181,103.941853,4,12
...,...,...,...,...,...
142,river valley high school,1.344078,103.709022,4(M),9(M)
143,"school of the arts, singapore",1.299577,103.848762,NaN,NaN
144,singapore sports school,1.426411,103.789445,NaN,NaN
145,st. joseph's institution,1.297258,103.851024,4,8


In [13]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(sec.LATITUDE.values, sec.LONGITUDE.values)

# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

count_rows = []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_sec_school": sec.NAME[nearest],
        "nearest_sec_school_m": round(float(d[nearest]), 1),
    }

    count_rows.append(results)
sec2 = sec[['NAME','pg3_min','pg3_max']]
sec2.columns = ['nearest_sec_school','sec_sch_score_entry_min','sec_sch_score_entry_max']
sec_counts = pd.DataFrame(count_rows)
sec_counts = pd.merge(sec_counts, sec2, how = 'left',on = 'nearest_sec_school')
sec_counts['sec_sch_score_entry_max'] = sec_counts['sec_sch_score_entry_max'].fillna('-1').apply(lambda x: int(x.split('(')[0]))
sec_counts['sec_sch_score_entry_min'] = sec_counts['sec_sch_score_entry_min'].fillna('-1').apply(lambda x: int(x.split('(')[0]))
sch_type2.columns = ['nearest_sec_school', 'sec_sch_type', 'sec_sch_nature']
sec_counts = pd.merge(sec_counts, sch_type2, how = 'left', on = ['nearest_sec_school'])

sec_counts

,TOWN,BLOCK,STREET,nearest_sec_school,nearest_sec_school_m,sec_sch_score_entry_min,sec_sch_score_entry_max,sec_sch_type,sec_sch_nature
0,serangoon,106,serangoon north avenue 1,serangoon garden secondary school,200.6,12,22,government school,co-ed school
1,bukit merah,34,telok blangah way,chij st. theresa's convent,204.0,9,20,government-aided sch,girls' school
2,punggol,226b,sumang lane,yusof ishak secondary school,558.0,7,15,government school,co-ed school
3,marine parade,74,marine drive,chij katong convent,1145.1,12,20,government-aided sch,girls' school
4,choa chu kang,284,choa chu kang avenue 3,bukit panjang govt. high school,505.3,7,11,government school,co-ed school
...,...,...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,chij secondary (toa payoh),260.3,8,20,government-aided sch,girls' school
9460,sembawang,307,canberra road,sembawang secondary school,539.4,18,22,government school,co-ed school
9461,woodlands,681a,woodlands drive 62,spectra secondary school,279.6,-1,-1,specialised independent school,co-ed school
9462,choa chu kang,673a,choa chu kang crescent,unity secondary school,200.6,12,18,government school,co-ed school


# Tertiary Schools
- [School Aggregations](#school-aggregations)
- [Nearest Schools](#nearest-schools)
- [Primary Schools](#primary-schools)
- [Secondary Schools](#secondary-schools)
- [Tertiary Schools](#tertiary-schools)
- [Conclusion](#conclusion)

In [14]:
jc = pd.read_csv(f'{ROOT}/data/features/raw/collected_data/school_data/sg-schools-junior_college.csv')
jc = jc[[
'NAME', 'LATITUDE', 'LONGITUDE', 'l1r5_arts_min', 'l1r5_arts_max', 
'l1r5_science_min', 'l1r5_science_max',
]]
jc = pd.concat([jc, mx_jc], axis = 0).reset_index(drop = True)
for col in ['l1r5_arts_min', 'l1r5_arts_max', 
'l1r5_science_min', 'l1r5_science_max',]:
    jc[col] = jc[col].fillna(-1).apply(lambda x: int(x))
jc

,NAME,LATITUDE,LONGITUDE,l1r5_arts_min,l1r5_arts_max,l1r5_science_min,l1r5_science_max
0,anderson serangoon junior college,1.365043,103.887039,7,10,3,9
1,anglo-chinese junior college,1.303043,103.785358,4,9,2,7
2,catholic junior college,1.324556,103.835029,5,12,6,11
3,eunoia junior college,1.362138,103.841286,2,6,2,5
4,jurong pioneer junior college,1.384374,103.754375,9,14,4,14
5,millennia institute,1.350319,103.741594,10,20,9,20
6,nanyang junior college,1.350283,103.866499,2,7,2,5
7,st andrew's junior college,1.333065,103.863576,5,10,3,8
8,tampines meridian junior college,1.376333,103.936991,7,12,7,12
9,victoria junior college,1.306702,103.919983,4,8,2,6


In [15]:
st_xy = to_xy(houses.LATITUDE.values, houses.LONGITUDE.values)
ml_xy = to_xy(jc.LATITUDE.values, jc.LONGITUDE.values)

# full pairwise distance matrix (192 x 113 — tiny)
dist = np.hypot(st_xy[:, 0][:, None] - ml_xy[:, 0][None, :],
                st_xy[:, 1][:, None] - ml_xy[:, 1][None, :])

count_rows = []
for s in range(len(houses)):
    results = {}
    d = dist[s]
    nearest = int(np.argmin(d))
    results = {
        "TOWN": houses.TOWN[s], "BLOCK": houses.BLOCK[s], "STREET": houses.STREET[s],
        "nearest_jc_school": jc.NAME[nearest],
        "nearest_jc_school_m": round(float(d[nearest]), 1),
    }

    count_rows.append(results)
jc2 = jc[['NAME', 'l1r5_arts_min', 'l1r5_arts_max', 'l1r5_science_min', 'l1r5_science_max']]
jc2.columns = ['nearest_jc_school','jc_arts_entry_min','jc_arts_entry_max','jc_science_entry_min','jc_science_entry_max']
jc_counts = pd.DataFrame(count_rows)
jc_counts = pd.merge(jc_counts, jc2, how = 'left', on = 'nearest_jc_school')
sch_type2.columns = ['nearest_jc_school', 'jc_sch_type', 'jc_sch_nature']
jc_counts = pd.merge(jc_counts, sch_type2, how = 'left', on = ['nearest_jc_school'])
jc_counts

,TOWN,BLOCK,STREET,nearest_jc_school,nearest_jc_school_m,jc_arts_entry_min,jc_arts_entry_max,jc_science_entry_min,jc_science_entry_max,jc_sch_type,jc_sch_nature
0,serangoon,106,serangoon north avenue 1,anderson serangoon junior college,1893.1,7,10,3,9,government school,co-ed school
1,bukit merah,34,telok blangah way,"school of the arts, singapore",4114.8,-1,-1,-1,-1,specialised school,co-ed school
2,punggol,226b,sumang lane,anderson serangoon junior college,4248.8,7,10,3,9,government school,co-ed school
3,marine parade,74,marine drive,victoria junior college,1458.0,4,8,2,6,government school,co-ed school
4,choa chu kang,284,choa chu kang avenue 3,jurong pioneer junior college,1455.6,9,14,4,14,government school,co-ed school
...,...,...,...,...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,catholic junior college,1489.1,5,12,6,11,government-aided sch,co-ed school
9460,sembawang,307,canberra road,yishun innova junior college,2581.7,5,20,9,18,government school,co-ed school
9461,woodlands,681a,woodlands drive 62,singapore sports school,2162.1,-1,-1,-1,-1,specialised school,co-ed school
9462,choa chu kang,673a,choa chu kang crescent,jurong pioneer junior college,2000.9,9,14,4,14,government school,co-ed school


In [ ]:
df_lst = [pri_counts, sec_counts, jc_counts]
sch_counts = counts
for df in df_lst:
    sch_counts = pd.merge(sch_counts, df, how = 'left', on = ['TOWN',"BLOCK", 'STREET'])
sch_counts

# Conclusion
- [School Aggregations](#school-aggregations)
- [Nearest Schools](#nearest-schools)
- [Primary Schools](#primary-schools)
- [Secondary Schools](#secondary-schools)
- [Tertiary Schools](#tertiary-schools)
- [Conclusion](#conclusion)

We produced three sets of tables relating to railway stations:
1. Nearest mall per railway station
2. Foot traffic per railway station
3. Number of railway stations per house

In [16]:
df_lst = [pri_counts, sec_counts, jc_counts]
sch_counts = counts
for df in df_lst:
    sch_counts = pd.merge(sch_counts, df, how = 'left', on = ['TOWN',"BLOCK", 'STREET'])
sch_counts

,TOWN,BLOCK,STREET,nearest_school,nearest_school_m,nearest_school_type,nearest_school_nature,nearest_school_level,nearest_pri_school,nearest_pri_school_m,...,sec_sch_type,sec_sch_nature,nearest_jc_school,nearest_jc_school_m,jc_arts_entry_min,jc_arts_entry_max,jc_science_entry_min,jc_science_entry_max,jc_sch_type,jc_sch_nature
0,serangoon,106,serangoon north avenue 1,serangoon garden secondary school,200.6,government school,co-ed school,secondary (s1-s5),rosyth school,549.8,...,government school,co-ed school,anderson serangoon junior college,1893.1,7,10,3,9,government school,co-ed school
1,bukit merah,34,telok blangah way,chij st. theresa's convent,204.0,government-aided sch,girls' school,secondary (s1-s5),radin mas primary school,248.3,...,government-aided sch,girls' school,"school of the arts, singapore",4114.8,-1,-1,-1,-1,specialised school,co-ed school
2,punggol,226b,sumang lane,yusof ishak secondary school,558.0,government school,co-ed school,secondary (s1-s5),punggol green primary school,584.2,...,government school,co-ed school,anderson serangoon junior college,4248.8,7,10,3,9,government school,co-ed school
3,marine parade,74,marine drive,chij (katong) primary,531.7,government-aided sch,girls' school,primary,chij (katong) primary,531.7,...,government-aided sch,girls' school,victoria junior college,1458.0,4,8,2,6,government school,co-ed school
4,choa chu kang,284,choa chu kang avenue 3,chua chu kang primary school,220.0,government school,co-ed school,primary,chua chu kang primary school,220.0,...,government school,co-ed school,jurong pioneer junior college,1455.6,9,14,4,14,government school,co-ed school
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9459,toa payoh,155,lorong 1 toa payoh,chij secondary (toa payoh),260.3,government-aided sch,girls' school,secondary (s1-s5),chij primary (toa payoh),416.0,...,government-aided sch,girls' school,catholic junior college,1489.1,5,12,6,11,government-aided sch,co-ed school
9460,sembawang,307,canberra road,sembawang primary school,120.3,government school,co-ed school,primary,sembawang primary school,120.3,...,government school,co-ed school,yishun innova junior college,2581.7,5,20,9,18,government school,co-ed school
9461,woodlands,681a,woodlands drive 62,greenwood primary school,110.9,government school,co-ed school,primary,greenwood primary school,110.9,...,specialised independent school,co-ed school,singapore sports school,2162.1,-1,-1,-1,-1,specialised school,co-ed school
9462,choa chu kang,673a,choa chu kang crescent,unity secondary school,200.6,government school,co-ed school,secondary (s1-s5),unity primary school,219.0,...,government school,co-ed school,jurong pioneer junior college,2000.9,9,14,4,14,government school,co-ed school


In [17]:
sch_counts[['TOWN', 'BLOCK', 'STREET', 'nearest_school', 'nearest_school_m',
       'nearest_school_type', 'nearest_school_nature', 'nearest_school_level',
       'nearest_pri_school', 'nearest_pri_school_m', 'pri_sch_type',
       'pri_sch_nature', 'nearest_sec_school', 'nearest_sec_school_m',
       'sec_sch_score_entry_min', 'sec_sch_score_entry_max', 'sec_sch_type',
       'sec_sch_nature', 'nearest_jc_school', 'nearest_jc_school_m',
       'jc_arts_entry_min', 'jc_arts_entry_max', 'jc_science_entry_min',
       'jc_science_entry_max', 'jc_sch_type', 'jc_sch_nature']].to_csv(
           f'{ROOT}/data/features/raw/location_data_mining/hdb_nearest_school.csv', index = False
       )